# Notebook 3 — Geometric Analysis

Near-complete reuse of the colours section from
`Reference_Papers/Representation-Manifolds/3-reproduce_figures.ipynb`.
SNOMED CT ontological distances replace hue distances.

**Input files** (from notebook 2)
- `{DATA_DIR}/embeddings_normalised.csv`
- `{DATA_DIR}/concepts_filtered.csv`
- `{DATA_DIR}/ontological_distances_filtered.csv`

In [ ]:
# parameters
DATA_DIR = "../../data/embeddings-concept-openai"

## 3a. Load

In [ ]:
from utils import knn_graph, largest_connected_component, interactive_3d_plot, interactive_3d_plot_by_tag, distance_plot

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import dijkstra
from utils import plot_scatter

In [ ]:
X        = pd.read_csv(f"{DATA_DIR}/embeddings_normalised.csv", header=None).values
concepts = pd.read_csv(f"{DATA_DIR}/concepts_filtered.csv")
D_snomed = pd.read_csv(f"{DATA_DIR}/ontological_distances_filtered.csv", header=None).values

print(f"X shape:        {X.shape}")
print(f"concepts shape: {concepts.shape}")
print(f"D_snomed shape: {D_snomed.shape}")

## 3b. k-NN graph and geodesic distances

In [ ]:
k = 5
A = knn_graph(X, k)
A.data[A.data < 1e-3] = 1e-3
lcc_mask = largest_connected_component(A, verbose=True)

In [ ]:
X_lcc        = X[lcc_mask]
D_snomed_lcc = D_snomed[np.ix_(lcc_mask, lcc_mask)]
concepts_lcc = concepts[lcc_mask].reset_index(drop=True)

A_lcc = knn_graph(X_lcc, k)
DXm   = dijkstra(A_lcc, return_predecessors=False)
DXc   = 1 - squareform(pdist(X_lcc, metric="cosine"))

print(f"LCC size: {X_lcc.shape[0]}")
print(f"Geodesic distances — finite: {np.isfinite(DXm).sum()}, infinite: {(~np.isfinite(DXm)).sum()}")

In [ ]:
from colorsys import hls_to_rgb
from matplotlib.patches import Patch

# Base hue per semantic tag (H in HLS, 0–1).  None → grey (unsaturated).
# Tags ordered by frequency in the "breast" search corpus.
TAG_HUE = {
    # Major groups
    "disorder":                   0.60,   # blue
    "procedure":                  0.00,   # red
    "finding":                    None,   # grey
    "observable entity":          0.08,   # orange
    "body structure":             0.48,   # teal
    "situation":                  0.35,   # green
    # Minor groups
    "physical object":            0.75,   # purple
    "specimen":                   0.53,   # cyan
    "organism":                   0.22,   # yellow-green
    "qualifier value":            0.90,   # pink
    "morphologic abnormality":    0.95,   # magenta-red
    "regime/therapy":             0.15,   # yellow
    "substance":                  0.45,   # blue-green
    "environment":                0.70,   # indigo
    "navigational concept":       0.78,   # violet
    "event":                      0.28,   # lime
    "assessment scale":           0.12,   # amber
    "tumor staging":              0.03,   # orange-red
    "record artifact":            0.88,   # rose
    "context-dependent category": 0.40,   # seafoam
    "occupation":                 0.10,   # gold
    "cell structure":             0.32,   # spring green
    "cell":                       0.25,   # chartreuse
    "staging scale":              0.58,   # sky blue
}

L_DARK, L_LIGHT = 0.25, 0.85   # lightness range (ancestor depth → lightness)
SATURATION = 0.75


def group_hex_colors(concepts_df, D_snomed):
    n = len(concepts_df)
    colors = ["#888888"] * n   # default grey for any tag not in TAG_HUE

    for tag in concepts_df["semantic_tag"].unique():
        hue  = TAG_HUE.get(tag, None)   # unknown tags fall back to grey
        mask = concepts_df["semantic_tag"] == tag
        idxs = np.where(mask)[0]
        if len(idxs) == 0:
            continue

        anc = concepts_df.loc[mask, "ancestor_count"].values.astype(float)
        mn, mx = anc.min(), anc.max()
        norm = (anc - mn) / (mx - mn + 1e-9)

        sat = 0 if hue is None else SATURATION
        h   = 0 if hue is None else hue

        for i, idx in enumerate(idxs):
            l = L_DARK + norm[i] * (L_LIGHT - L_DARK)
            r, g, b = hls_to_rgb(h, l, sat)
            colors[idx] = "#{:02x}{:02x}{:02x}".format(
                int(r*255), int(g*255), int(b*255))
    return colors


def legend_handles(concepts_df=None):
    """Patch handles for all semantic tags present in the data, sorted by frequency.
    Pass concepts_df to restrict to tags actually in the dataset; omit to show all TAG_HUE entries.
    """
    L_mid = (L_DARK + L_LIGHT) / 2
    if concepts_df is not None:
        tag_counts = concepts_df["semantic_tag"].value_counts()
        tags = tag_counts.index.tolist()
    else:
        tags = list(TAG_HUE.keys())

    handles = []
    for tag in tags:
        hue = TAG_HUE.get(tag, None)
        h   = 0    if hue is None else hue
        sat = 0    if hue is None else SATURATION
        r, g, b = hls_to_rgb(h, L_mid, sat)
        color = "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))
        handles.append(Patch(facecolor=color, label=tag))
    return handles


hex_colors = group_hex_colors(concepts_lcc, D_snomed_lcc)
print(f"hex_colors computed for {len(hex_colors)} LCC concepts")

## 3c. PCA visualisation

In [ ]:
pca = PCA(n_components=3)
Xp  = pca.fit_transform(X_lcc)

print(f"Explained variance ratio (3 PCs): {pca.explained_variance_ratio_.sum():.4f}")

fig = interactive_3d_plot(
    Xp,
    labels=concepts_lcc["preferred_term"].values,
    color_values=hex_colors,
    colormap=None,
    point_size=3,
)
fig.write_html(f"{DATA_DIR}/interactive_3d.html", include_plotlyjs=True)
fig.show()

### 3c-ii. PCA — interactive by semantic tag

In [ ]:
fig2 = interactive_3d_plot_by_tag(
    Xp,
    labels=concepts_lcc["preferred_term"].values,
    colors=hex_colors,
    groups=concepts_lcc["semantic_tag"].values,
    point_size=3,
)
fig2.write_html(f"{DATA_DIR}/interactive_3d_by_tag.html", include_plotlyjs=True)
fig2.show()

## 3d. Correlation plots

In [ ]:
# Manifold distance vs SNOMED ontological distance
fig, ax = distance_plot(
    DXm, D_snomed_lcc,
    labels=concepts_lcc["preferred_term"].values,
    colors=hex_colors,
    corr_coef="pearson",
    xlabel="SNOMED CT ontological distance",
    ylabel="Manifold distance",
)
ax.legend(handles=legend_handles(concepts_lcc), title="Semantic tag", loc="upper left")
fig.suptitle("Manifold distance vs SNOMED CT ontological distance", y=1.02)
fig.savefig(f"{DATA_DIR}/plot_manifold_vs_ontological.png", dpi=300, bbox_inches="tight")

In [ ]:
threshold = 30

xs_f, ys_f, ls_f = [], [], []
for i in range(len(D_snomed_lcc)):
    row = D_snomed_lcc[i]
    for d in np.unique(row[row < threshold]):
        xs_f.append(d)
        ys_f.append(np.mean(DXm[i, row == d]))
        ls_f.append(hex_colors[i])

xs_f = np.array(xs_f)
ys_f = np.array(ys_f)
ls_f = np.array(ls_f)

idx = np.random.permutation(len(xs_f))
xs_f, ys_f, ls_f = xs_f[idx], ys_f[idx], ls_f[idx]

corr = np.corrcoef(xs_f, ys_f)[0, 1]

fig, ax = plot_scatter(
    xs_f, ys_f, alpha=0.1, marker_size=0.1,
    xlabel="SNOMED CT ontological distance",
    ylabel="Manifold distance",
    colors=ls_f,
    y_min=min(ys_f),
    text_box=fr"$\rho = {corr:.3f}$",
    figsize=(3.5, 3.5),
)
ax.legend(handles=legend_handles(concepts_lcc), title="Semantic tag", loc="upper left")
fig.suptitle("Manifold distance vs SNOMED CT ontological distance\n(ontological distance < 30)", y=1.02)
fig.savefig(f"{DATA_DIR}/plot_manifold_vs_ontological_lt30.png", dpi=300, bbox_inches="tight")

In [ ]:
# Cosine similarity vs SNOMED ontological distance
fig, ax = distance_plot(
    DXc, D_snomed_lcc,
    labels=concepts_lcc["preferred_term"].values,
    colors=hex_colors,
    corr_coef="chatterjee",
    square_distances=True,
    xlabel="Squared SNOMED CT ontological distance",
    ylabel="Cosine similarity",
)
ax.legend(
    handles=legend_handles(concepts_lcc), title="Semantic tag",
    loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0,
)
fig.suptitle("Cosine similarity vs SNOMED CT ontological distance²", y=1.02)
fig.savefig(f"{DATA_DIR}/plot_cosine_vs_ontological.png", dpi=300, bbox_inches="tight")